# ver7 환자 단위 조기 탐지 - Validation Threshold Calibration

기존 full fine-tuning 5-fold checkpoint를 재사용하여, validation 환자에서 조기 탐지 목적의 threshold를 선택하고 outer test에서 평가합니다.

CNN 재학습은 수행하지 않습니다.

## 1. 환경 설정

In [1]:
import os
import re
import gc
import sys
import time
import random
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager

import numpy as np
import pandas as pd
import torch

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.executable}")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"device: {device}")

Python: C:\Users\user\anaconda3\envs\alzheimer\python.exe
torch: 2.12.0.dev20260408+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070
device: cuda


## 2. 실험 Config

In [2]:
DATA_ROOT = Path(r"C:\Users\user\Desktop\alzheimer_dataset\Data")
OUTPUT_DIR = Path(r"C:\Users\user\alzheimer\patient_level_cv")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

NEGATIVE_CLASS = "NonDemented"
POSITIVE_CLASS = "VeryMildDemented"
TASK_NAME = "non_vs_verymild"

N_SPLITS = 5
INNER_VAL_RATIO = 0.15

# 전체 비교는 두 전략 × 5 folds = 10회 학습입니다.
EXPERIMENTS_TO_RUN = ["full_finetune", "partial_finetune"]
FOLDS_TO_RUN = [1, 2, 3, 4, 5]

# 실행 확인용으로 시간을 줄이려면 아래처럼 바꿀 수 있습니다.
# EXPERIMENTS_TO_RUN = ["full_finetune", "partial_finetune"]
# FOLDS_TO_RUN = [1]

TRAIN_AUG_MODE = "sepaug_4n"  # "original", "sepaug_3n", "sepaug_4n"
ROTATION_DEGREES = 10
SHIFT_TRANSLATE = (0.05, 0.05)
ZOOM_SCALE = (0.9, 1.1)
DETERMINISTIC_AUGMENTATION = True

BATCH_SIZE = 32
NUM_WORKERS = 0
PIN_MEMORY = True
PERSISTENT_WORKERS = NUM_WORKERS > 0
USE_AMP = True
WEIGHT_DECAY = 1e-4
BALANCE_STRATEGY = "class_weight_sqrt"
EARLY_STOPPING_MIN_DELTA = 1e-4

# ver4와 ver5 설정을 그대로 비교합니다.
EXPERIMENT_CONFIGS = {
    "full_finetune": {
        "epochs": 12,
        "freeze_epochs": 3,
        "head_lr": 1e-3,
        "backbone_lr": 1e-5,
        "patience": 3,
        "unfreeze_mode": "all",
    },
    "partial_finetune": {
        "epochs": 8,
        "freeze_epochs": 3,
        "head_lr": 3e-4,
        "backbone_lr": 3e-6,
        "patience": 2,
        "unfreeze_mode": "last_stages",
    },
}

print(f"Task: {NEGATIVE_CLASS} vs {POSITIVE_CLASS}")
print(f"Outer folds: {N_SPLITS}")
print(f"Experiments: {EXPERIMENTS_TO_RUN}")
print(f"Folds to run: {FOLDS_TO_RUN}")

Task: NonDemented vs VeryMildDemented
Outer folds: 5
Experiments: ['full_finetune', 'partial_finetune']
Folds to run: [1, 2, 3, 4, 5]


## 3. 환자 Manifest 생성

In [3]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
PATIENT_PATTERN = re.compile(r"^(OAS1_\d+)", re.IGNORECASE)

rows = []
for target, class_name in enumerate([NEGATIVE_CLASS, POSITIVE_CLASS]):
    class_dir = DATA_ROOT / class_name
    assert class_dir.exists(), class_dir
    for image_path in sorted(class_dir.iterdir()):
        if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        patient_match = PATIENT_PATTERN.match(image_path.name)
        assert patient_match, image_path.name
        rows.append({
            "image_path": str(image_path),
            "class_name": class_name,
            "patient_id": patient_match.group(1).upper(),
            "target": target,
        })

manifest = pd.DataFrame(rows)
assert manifest.groupby("patient_id")["target"].nunique().max() == 1

patient_table = (
    manifest.groupby("patient_id", as_index=False)
    .agg(target=("target", "first"), class_name=("class_name", "first"), image_count=("image_path", "count"))
)

print(f"Images: {len(manifest):,}")
print(f"Patients: {len(patient_table):,}")
print(patient_table["target"].value_counts().sort_index())

Images: 80,947
Patients: 324
target
0    266
1     58
Name: count, dtype: int64


## 4. Dataset 및 환자 단위 평가 함수

In [4]:
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

weights = models.EfficientNet_B0_Weights.DEFAULT
preprocess = weights.transforms()

@contextmanager
def temporary_seed(seed):
    py_state = random.getstate()
    np_state = np.random.get_state()
    torch_state = torch.random.get_rng_state()
    try:
        random.seed(seed)
        np.random.seed(seed % (2**32 - 1))
        torch.manual_seed(seed)
        yield
    finally:
        random.setstate(py_state)
        np.random.set_state(np_state)
        torch.random.set_rng_state(torch_state)

class PatientSliceDataset(Dataset):
    def __init__(self, dataframe, preprocess, aug_mode="original", deterministic=True, seed=42):
        self.df = dataframe.reset_index(drop=True).copy()
        self.preprocess = preprocess
        self.deterministic = deterministic
        self.seed = seed

        if aug_mode == "original":
            self.output_types = ["original"]
        elif aug_mode == "sepaug_3n":
            self.output_types = ["rotation", "shift", "zoom"]
        elif aug_mode == "sepaug_4n":
            self.output_types = ["original", "rotation", "shift", "zoom"]
        else:
            raise ValueError(aug_mode)

        self.rotation = transforms.RandomRotation(ROTATION_DEGREES)
        self.shift = transforms.RandomAffine(degrees=0, translate=SHIFT_TRANSLATE)
        self.zoom = transforms.RandomAffine(degrees=0, scale=ZOOM_SCALE)

    def __len__(self):
        return len(self.df) * len(self.output_types)

    def _augment(self, image, aug_type):
        if aug_type == "original":
            return image
        if aug_type == "rotation":
            return self.rotation(image)
        if aug_type == "shift":
            return self.shift(image)
        if aug_type == "zoom":
            return self.zoom(image)
        raise ValueError(aug_type)

    def __getitem__(self, idx):
        multiplier = len(self.output_types)
        base_idx = idx // multiplier
        aug_idx = idx % multiplier
        row = self.df.iloc[base_idx]

        image = Image.open(row["image_path"]).convert("RGB")
        if self.deterministic:
            with temporary_seed(self.seed + base_idx * 1009 + aug_idx * 9176):
                image = self._augment(image, self.output_types[aug_idx])
        else:
            image = self._augment(image, self.output_types[aug_idx])

        return (
            self.preprocess(image),
            torch.tensor(int(row["target"]), dtype=torch.long),
            row["patient_id"],
        )

def build_model():
    model = models.efficientnet_b0(weights=weights)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
    return model.to(device)

def aggregate_patient_predictions(patient_ids, labels, probabilities, threshold=0.5):
    grouped_probs = defaultdict(list)
    grouped_labels = {}
    for patient_id, label, prob in zip(patient_ids, labels, probabilities):
        grouped_probs[patient_id].append(prob)
        grouped_labels[patient_id] = label
    ordered_ids = sorted(grouped_probs)
    patient_probs = np.array([np.mean(grouped_probs[pid]) for pid in ordered_ids])
    patient_labels = np.array([grouped_labels[pid] for pid in ordered_ids])
    patient_preds = (patient_probs >= threshold).astype(int)
    return ordered_ids, patient_labels, patient_probs, patient_preds

def calculate_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    specificity = cm[0, 0] / max(cm[0].sum(), 1)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "auroc": roc_auc_score(y_true, y_prob),
        "auprc": average_precision_score(y_true, y_prob),
    }

## 5. Fine-Tuning 비교 함수

In [5]:
from sklearn.model_selection import StratifiedKFold, train_test_split

def configure_trainable_layers(model, mode):
    for parameter in model.parameters():
        parameter.requires_grad = False

    if mode == "head_only":
        pass
    elif mode == "all":
        for parameter in model.features.parameters():
            parameter.requires_grad = True
    elif mode == "last_stages":
        for parameter in model.features[7].parameters():
            parameter.requires_grad = True
        for parameter in model.features[8].parameters():
            parameter.requires_grad = True
    else:
        raise ValueError(mode)

    for parameter in model.classifier.parameters():
        parameter.requires_grad = True

def make_optimizer(model, config):
    head_ids = {id(p) for p in model.classifier.parameters()}
    head_params, backbone_params = [], []
    for parameter in model.parameters():
        if not parameter.requires_grad:
            continue
        if id(parameter) in head_ids:
            head_params.append(parameter)
        else:
            backbone_params.append(parameter)
    groups = []
    if backbone_params:
        groups.append({"params": backbone_params, "lr": config["backbone_lr"]})
    groups.append({"params": head_params, "lr": config["head_lr"]})
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)

def make_class_weight(inner_train_patients):
    counts_series = inner_train_patients["target"].value_counts().sort_index()
    counts = np.array([counts_series.get(0, 0), counts_series.get(1, 0)], dtype=np.float64)
    if BALANCE_STRATEGY == "class_weight_sqrt":
        values = 1.0 / np.sqrt(np.maximum(counts, 1))
        values /= values.mean()
        return torch.tensor(values, dtype=torch.float32, device=device)
    return None

def evaluate_model(model, loader, threshold=0.5):
    model.eval()
    patient_ids, labels_all, probabilities_all = [], [], []
    with torch.inference_mode():
        for images, labels, batch_patient_ids in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and torch.cuda.is_available())):
                logits = model(images)
            probabilities = logits.softmax(dim=1)[:, 1].cpu().numpy()
            patient_ids.extend(list(batch_patient_ids))
            labels_all.extend(labels.numpy().tolist())
            probabilities_all.extend(probabilities.tolist())

    ids, y_true, y_prob, y_pred = aggregate_patient_predictions(
        patient_ids, labels_all, probabilities_all, threshold=threshold
    )
    return calculate_metrics(y_true, y_prob, threshold), ids, y_true, y_prob

def train_one_fold(experiment_name, fold_number, outer_train_patients, outer_test_patients):
    config = EXPERIMENT_CONFIGS[experiment_name]
    run_seed = SEED + fold_number * 100
    seed_everything(run_seed)

    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    inner_train_ids = set(inner_train_patients["patient_id"])
    inner_val_ids = set(inner_val_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])
    assert inner_train_ids.isdisjoint(inner_val_ids)
    assert inner_train_ids.isdisjoint(outer_test_ids)
    assert inner_val_ids.isdisjoint(outer_test_ids)

    train_df = manifest[manifest["patient_id"].isin(inner_train_ids)].copy()
    val_df = manifest[manifest["patient_id"].isin(inner_val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(outer_test_ids)].copy()

    train_dataset = PatientSliceDataset(
        train_df, preprocess, TRAIN_AUG_MODE, DETERMINISTIC_AUGMENTATION, run_seed
    )
    val_dataset = PatientSliceDataset(val_df, preprocess, "original", True, run_seed)
    test_dataset = PatientSliceDataset(test_df, preprocess, "original", True, run_seed)

    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )
    test_loader = DataLoader(
        test_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )

    model = build_model()
    configure_trainable_layers(model, "head_only")
    optimizer = make_optimizer(model, config)
    class_weight = make_class_weight(inner_train_patients)
    criterion = nn.CrossEntropyLoss(weight=class_weight)
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and torch.cuda.is_available()))

    best_val_auroc = -1.0
    best_epoch = 0
    best_state = None
    epochs_without_improvement = 0
    fine_tuning_started = False

    print(
        f"\n[{experiment_name} fold {fold_number}] "
        f"inner train={len(inner_train_patients)}, val={len(inner_val_patients)}, "
        f"outer test={len(outer_test_patients)}"
    )

    for epoch in range(1, config["epochs"] + 1):
        if epoch == config["freeze_epochs"] + 1:
            configure_trainable_layers(model, config["unfreeze_mode"])
            optimizer = make_optimizer(model, config)
            fine_tuning_started = True
            epochs_without_improvement = 0

        model.train()
        loss_sum, total = 0.0, 0
        for images, labels, patient_ids in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(USE_AMP and torch.cuda.is_available())):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item() * labels.size(0)
            total += labels.size(0)

        val_metrics, _, _, _ = evaluate_model(model, val_loader)
        improved = val_metrics["auroc"] > best_val_auroc + EARLY_STOPPING_MIN_DELTA
        if improved:
            best_val_auroc = val_metrics["auroc"]
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        elif fine_tuning_started:
            epochs_without_improvement += 1

        print(
            f"Epoch {epoch:02d}/{config['epochs']} | loss {loss_sum / max(total, 1):.4f} | "
            f"val AUROC {val_metrics['auroc']:.4f} AUPRC {val_metrics['auprc']:.4f} | "
            f"early {epochs_without_improvement}/{config['patience']}"
        )

        if fine_tuning_started and epochs_without_improvement >= config["patience"]:
            print("Early stopping")
            break

    assert best_state is not None
    model.load_state_dict(best_state)
    model = model.to(device)

    test_metrics, test_ids, y_true, y_prob = evaluate_model(model, test_loader)
    checkpoint_path = CHECKPOINT_DIR / f"{TASK_NAME}_{experiment_name}_fold{fold_number}.pt"
    torch.save({
        "model_state_dict": best_state,
        "experiment": experiment_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "best_val_auroc": best_val_auroc,
        "test_metrics": test_metrics,
    }, checkpoint_path)

    result = {
        "experiment": experiment_name,
        "fold": fold_number,
        "best_epoch": best_epoch,
        "val_auroc": best_val_auroc,
        "test_patients": len(test_ids),
        **test_metrics,
    }
    print("Test:", result)

    del model, optimizer, scaler, train_loader, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

## 6. Validation 기반 Threshold Calibration

이 셀은 CNN을 다시 학습하지 않습니다. 기존 `full_finetune` 5-fold checkpoint를 불러옵니다.

각 fold에서:

1. Outer train 환자 안에서 기존과 동일한 inner validation 환자를 재구성합니다.
2. Validation 환자의 예측 확률에서 sensitivity가 목표값 이상인 threshold를 찾습니다.
3. 조건을 만족하는 threshold 중 specificity가 가장 높은 값을 선택합니다.
4. 선택된 threshold를 해당 outer test fold에 그대로 적용합니다.

Outer test의 label이나 확률은 threshold 선택에 사용하지 않습니다.

In [6]:
from sklearn.model_selection import StratifiedKFold, train_test_split

TARGET_VALIDATION_SENSITIVITY = 0.80
CALIBRATION_EXPERIMENT = "full_finetune"

def choose_threshold_by_sensitivity(y_true, y_prob, target_sensitivity=0.80):
    """목표 sensitivity를 만족하면서 specificity가 가장 높은 threshold를 선택합니다."""
    candidates = np.unique(np.concatenate([
        np.array([0.0, 0.5, 1.0]),
        np.asarray(y_prob, dtype=np.float64),
    ]))

    rows = []
    for threshold in candidates:
        metrics = calculate_metrics(y_true, y_prob, threshold=float(threshold))
        rows.append({"threshold": float(threshold), **metrics})

    threshold_table = pd.DataFrame(rows)
    eligible = threshold_table[
        threshold_table["sensitivity"] >= target_sensitivity
    ].copy()

    if eligible.empty:
        # 목표 sensitivity를 달성할 수 없다면 sensitivity가 가장 높은 지점을 사용합니다.
        eligible = threshold_table[
            threshold_table["sensitivity"] == threshold_table["sensitivity"].max()
        ].copy()

    # specificity가 가장 높은 지점 우선.
    # 동률이면 threshold가 높은 지점을 선택해 불필요한 false positive를 줄입니다.
    selected = eligible.sort_values(
        ["specificity", "precision", "threshold"],
        ascending=[False, False, False],
    ).iloc[0]

    return float(selected["threshold"]), threshold_table

def build_fold_dataframes(outer_train_patients, outer_test_patients, fold_number):
    run_seed = SEED + fold_number * 100
    inner_train_patients, inner_val_patients = train_test_split(
        outer_train_patients,
        test_size=INNER_VAL_RATIO,
        random_state=run_seed,
        stratify=outer_train_patients["target"],
    )

    inner_train_ids = set(inner_train_patients["patient_id"])
    inner_val_ids = set(inner_val_patients["patient_id"])
    outer_test_ids = set(outer_test_patients["patient_id"])

    assert inner_train_ids.isdisjoint(inner_val_ids)
    assert inner_train_ids.isdisjoint(outer_test_ids)
    assert inner_val_ids.isdisjoint(outer_test_ids)

    val_df = manifest[manifest["patient_id"].isin(inner_val_ids)].copy()
    test_df = manifest[manifest["patient_id"].isin(outer_test_ids)].copy()
    return val_df, test_df

def make_original_loader(dataframe, seed):
    dataset = PatientSliceDataset(
        dataframe,
        preprocess,
        aug_mode="original",
        deterministic=True,
        seed=seed,
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=PERSISTENT_WORKERS,
    )

outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
patient_indices = np.arange(len(patient_table))
patient_targets = patient_table["target"].to_numpy()

calibrated_results = []
threshold_tables = {}

for fold_number, (outer_train_idx, outer_test_idx) in enumerate(
    outer_cv.split(patient_indices, patient_targets), start=1
):
    outer_train_patients = patient_table.iloc[outer_train_idx].reset_index(drop=True)
    outer_test_patients = patient_table.iloc[outer_test_idx].reset_index(drop=True)
    val_df, test_df = build_fold_dataframes(
        outer_train_patients,
        outer_test_patients,
        fold_number,
    )

    run_seed = SEED + fold_number * 100
    val_loader = make_original_loader(val_df, run_seed)
    test_loader = make_original_loader(test_df, run_seed)

    checkpoint_path = (
        CHECKPOINT_DIR /
        f"{TASK_NAME}_{CALIBRATION_EXPERIMENT}_fold{fold_number}.pt"
    )
    assert checkpoint_path.exists(), f"Checkpoint not found: {checkpoint_path}"

    # 이 checkpoint는 ver6에서 직접 생성한 신뢰 가능한 로컬 파일입니다.
    # PyTorch 2.6+는 torch.load의 weights_only 기본값이 True이므로,
    # NumPy scalar가 포함된 checkpoint dictionary를 읽으려면 False를 명시해야 합니다.
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    model = build_model()
    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    _, val_ids, val_y_true, val_y_prob = evaluate_model(
        model, val_loader, threshold=0.5
    )
    selected_threshold, threshold_table = choose_threshold_by_sensitivity(
        val_y_true,
        val_y_prob,
        target_sensitivity=TARGET_VALIDATION_SENSITIVITY,
    )
    threshold_tables[fold_number] = threshold_table

    val_metrics = calculate_metrics(
        val_y_true,
        val_y_prob,
        threshold=selected_threshold,
    )

    _, test_ids, test_y_true, test_y_prob = evaluate_model(
        model, test_loader, threshold=0.5
    )
    test_metrics_default = calculate_metrics(
        test_y_true,
        test_y_prob,
        threshold=0.5,
    )
    test_metrics_calibrated = calculate_metrics(
        test_y_true,
        test_y_prob,
        threshold=selected_threshold,
    )

    result = {
        "fold": fold_number,
        "best_epoch": checkpoint.get("best_epoch"),
        "selected_threshold": selected_threshold,
        "val_patients": len(val_ids),
        "test_patients": len(test_ids),
        "val_sensitivity": val_metrics["sensitivity"],
        "val_specificity": val_metrics["specificity"],
        "default_test_sensitivity": test_metrics_default["sensitivity"],
        "default_test_specificity": test_metrics_default["specificity"],
        "default_test_f1": test_metrics_default["f1"],
        "calibrated_test_accuracy": test_metrics_calibrated["accuracy"],
        "calibrated_test_precision": test_metrics_calibrated["precision"],
        "calibrated_test_sensitivity": test_metrics_calibrated["sensitivity"],
        "calibrated_test_specificity": test_metrics_calibrated["specificity"],
        "calibrated_test_f1": test_metrics_calibrated["f1"],
        "calibrated_test_macro_f1": test_metrics_calibrated["macro_f1"],
        "test_auroc": test_metrics_calibrated["auroc"],
        "test_auprc": test_metrics_calibrated["auprc"],
    }
    calibrated_results.append(result)

    print(
        f"Fold {fold_number} | threshold={selected_threshold:.4f} | "
        f"val sens/spec={val_metrics['sensitivity']:.3f}/{val_metrics['specificity']:.3f} | "
        f"test default sens/spec={test_metrics_default['sensitivity']:.3f}/{test_metrics_default['specificity']:.3f} | "
        f"test calibrated sens/spec={test_metrics_calibrated['sensitivity']:.3f}/{test_metrics_calibrated['specificity']:.3f}"
    )

    del model, val_loader, test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

calibrated_results_df = pd.DataFrame(calibrated_results)
calibrated_results_path = OUTPUT_DIR / "full_finetune_threshold_calibration_results.csv"
calibrated_results_df.to_csv(
    calibrated_results_path,
    index=False,
    encoding="utf-8-sig",
)

print(f"\n결과 저장: {calibrated_results_path}")
display(calibrated_results_df)

Fold 1 | threshold=0.7288 | val sens/spec=0.857/1.000 | test default sens/spec=0.545/0.926 | test calibrated sens/spec=0.182/0.963
Fold 2 | threshold=0.2663 | val sens/spec=0.857/0.875 | test default sens/spec=0.333/0.962 | test calibrated sens/spec=0.833/0.906
Fold 3 | threshold=0.3064 | val sens/spec=1.000/0.719 | test default sens/spec=0.333/0.925 | test calibrated sens/spec=0.917/0.698
Fold 4 | threshold=0.2541 | val sens/spec=1.000/0.875 | test default sens/spec=0.333/0.868 | test calibrated sens/spec=0.667/0.774
Fold 5 | threshold=0.4419 | val sens/spec=0.857/0.875 | test default sens/spec=0.818/0.830 | test calibrated sens/spec=1.000/0.792

결과 저장: C:\Users\user\alzheimer\patient_level_cv\full_finetune_threshold_calibration_results.csv


,fold,best_epoch,selected_threshold,val_patients,test_patients,val_sensitivity,val_specificity,default_test_sensitivity,default_test_specificity,default_test_f1,calibrated_test_accuracy,calibrated_test_precision,calibrated_test_sensitivity,calibrated_test_specificity,calibrated_test_f1,calibrated_test_macro_f1,test_auroc,test_auprc
0,1,5,0.728807,39,65,0.857143,1.00000,0.545455,0.925926,0.571429,0.830769,0.500000,0.181818,0.962963,0.266667,0.585507,0.873737,0.558959
1,2,4,0.266349,39,65,0.857143,0.87500,0.333333,0.962264,0.444444,0.892308,0.666667,0.833333,0.905660,0.740741,0.836390,0.933962,0.720217
2,3,3,0.306362,39,65,1.000000,0.71875,0.333333,0.924528,0.400000,0.738462,0.407407,0.916667,0.698113,0.564103,0.688645,0.853774,0.544025
3,4,4,0.254129,39,65,1.000000,0.87500,0.333333,0.867925,0.347826,0.753846,0.400000,0.666667,0.773585,0.500000,0.668367,0.838050,0.529096
4,5,1,0.441852,39,64,0.857143,0.87500,0.818182,0.830189,0.620690,0.828125,0.500000,1.000000,0.792453,0.666667,0.775439,0.909091,0.626170


## 7. Threshold 0.5 대비 Calibration 결과 요약

In [7]:
summary_metrics = [
    "default_test_sensitivity",
    "default_test_specificity",
    "default_test_f1",
    "calibrated_test_sensitivity",
    "calibrated_test_specificity",
    "calibrated_test_f1",
    "calibrated_test_macro_f1",
    "test_auroc",
    "test_auprc",
    "selected_threshold",
]

summary_rows = []
for metric in summary_metrics:
    summary_rows.append({
        "metric": metric,
        "mean": calibrated_results_df[metric].mean(),
        "std": calibrated_results_df[metric].std(ddof=1),
    })

calibration_summary_df = pd.DataFrame(summary_rows)
calibration_summary_path = OUTPUT_DIR / "full_finetune_threshold_calibration_summary.csv"
calibration_summary_df.to_csv(
    calibration_summary_path,
    index=False,
    encoding="utf-8-sig",
)

display(calibration_summary_df)

default_sens = calibrated_results_df["default_test_sensitivity"].mean()
calibrated_sens = calibrated_results_df["calibrated_test_sensitivity"].mean()
default_spec = calibrated_results_df["default_test_specificity"].mean()
calibrated_spec = calibrated_results_df["calibrated_test_specificity"].mean()

print(
    f"평균 sensitivity: {default_sens:.4f} -> {calibrated_sens:.4f} "
    f"({calibrated_sens - default_sens:+.4f})"
)
print(
    f"평균 specificity: {default_spec:.4f} -> {calibrated_spec:.4f} "
    f"({calibrated_spec - default_spec:+.4f})"
)
print(
    f"선택 threshold 평균: "
    f"{calibrated_results_df['selected_threshold'].mean():.4f} ± "
    f"{calibrated_results_df['selected_threshold'].std(ddof=1):.4f}"
)
print(f"summary saved: {calibration_summary_path}")

,metric,mean,std
0,default_test_sensitivity,0.472727,0.213846
1,default_test_specificity,0.902166,0.052511
2,default_test_f1,0.476878,0.115367
3,calibrated_test_sensitivity,0.719697,0.324964
4,calibrated_test_specificity,0.826555,0.106455
5,calibrated_test_f1,0.547635,0.182306
6,calibrated_test_macro_f1,0.710870,0.097392
7,test_auroc,0.881723,0.039466
8,test_auprc,0.595693,0.078882
9,selected_threshold,0.399500,0.198604


평균 sensitivity: 0.4727 -> 0.7197 (+0.2470)
평균 specificity: 0.9022 -> 0.8266 (-0.0756)
선택 threshold 평균: 0.3995 ± 0.1986
summary saved: C:\Users\user\alzheimer\patient_level_cv\full_finetune_threshold_calibration_summary.csv


## 8. 해석 기준

- Calibration 후 sensitivity가 증가하고 specificity가 허용 가능한 범위라면 조기 탐지용 threshold를 사용합니다.
- AUROC와 AUPRC는 threshold와 무관하므로 기존 값과 동일해야 합니다.
- Fold별 threshold 차이가 너무 크면 validation 양성 환자 수가 적어 threshold가 불안정한 것입니다.
- 최종 운영 threshold는 5개 test 결과를 보고 다시 선택하지 않습니다. 각 fold validation에서 선택된 threshold의 분포와 성능을 보고, 향후 독립 validation cohort에서 고정해야 합니다.